### How to Parse 10-K Report from EDGAR (SEC)
https://gist.github.com/anshoomehra/ead8925ea291e233a5aa2dcaa2dc61b2

## Intro
In this notebook we will apply REGEX & BeautifulSoup to find useful financial information in 10-Ks. In particular, we will extract text from Items 1A, 7, and 7A of 10-K.

## STEP 1 : Import Libraries
Note that, we will need parser for BeautifulSoup, there are many parsers, we will be using 'lxml' which can be pre-installed as follows & it help BeatifulSoup read HTML, XML documents:

In [12]:
#!pip install lxml

In [13]:
import requests # Import requests to retrive Web Urls example HTML. TXT 
from bs4 import BeautifulSoup # Import BeautifulSoup
import re # import re module for REGEXes
import pandas as pd # import pandas

## STEP 2 : Get Apple's [AAPL] 2018 10-K
Though we are using AAPL as example 10-K here, the pipeline being built is generic & can be used for other companies 10-K

SEC Website URL for 10-K (TEXT version)

SEC Website URL for 10-K (HTML version) It will be good to view/study along in html format to see how the below code would apply.

All the documents can be easily ssearched via CIK or company details via SEC's search tool

In [14]:
# Get the HTML data from the 2018 10-K from Apple
r = requests.get('https://www.sec.gov/Archives/edgar/data/320193/000032019318000145/0000320193-18-000145.txt')
raw_10k = r.text

If we print the raw_10k string we will see that it has many sections. In the code below, we print part of the raw_10k string:

In [15]:
print(raw_10k[0:1300])

<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html xmlns="http://www.w3.org/1999/xhtml">
<head>
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<title>SEC.gov | Request Rate Threshold Exceeded</title>
<style>
html {height: 100%}
body {height: 100%; margin:0; padding:0;}
#header {background-color:#003968; color:#fff; padding:15px 20px 10px 20px;font-family:Arial, Helvetica, sans-serif; font-size:20px; border-bottom:solid 5px #000;}
#footer {background-color:#003968; color:#fff; padding:15px 20px;font-family:Arial, Helvetica, sans-serif; font-size:20px;}
#content {max-width:650px;margin:60px auto; padding:0 20px 100px 20px; background-image:url(seal_bw.png);background-repeat:no-repeat;background-position:50% 100%;}
h1 {font-family:Georgia, Times, serif; font-size:20px;}
h2 {text-align:center; font-family:Georgia, Times, serif; font-size:20px; width:100%; border-bottom:solid #999 1px;padding

## STEP 3 : Apply REGEXes to find 10-K Section from the document

For our purposes, we are only interested in the sections that contain the 10-K information. All the sections, including the 10-K are contained within the <DOCUMENT> and </DOCUMENT> tags. Each section within the document tags is clearly marked by a <TYPE> tag followed by the name of the section.

In [24]:
# Regex to find <DOCUMENT> tags
doc_start_pattern = re.compile(r'<DOCUMENT>')
doc_end_pattern   = re.compile(r'</DOCUMENT>')
# Regex to find <TYPE> tag prceeding any characters, terminating at new line
type_pattern      = re.compile(r'<TYPE>[^\n]+')

Define Span Indices using REGEXes

Now, that we have the regexes defined, we will use the .finditer() method to match the regexes in the raw_10k. In the code below, we will create 3 lists:

A list that holds the .end() index of each match of doc_start_pattern

A list that holds the .start() index of each match of doc_end_pattern

A list that holds the name of section from each match of type_pattern

In [17]:
# Create 3 lists with the span indices for each regex

### There are many <Document> Tags in this text file, each as specific exhibit like 10-K, EX-10.17 etc
### First filter will give us document tag start <end> and document tag end's <start> 
### We will use this to later grab content in between these tags
doc_start_is = [x.end()           for x in doc_start_pattern.finditer(raw_10k)]
doc_end_is   = [x.start()         for x in   doc_end_pattern.finditer(raw_10k)]

### Type filter is interesting, it looks for <TYPE> with Not flag as new line, ie terminare there, with + sign
### to look for any char afterwards until new line \n. This will give us <TYPE> followed Section Name like '10-K'
### Once we have have this, it returns String Array, below line will with find content after <TYPE> ie, '10-K' 
### as section names
doc_types    = [x[len('<TYPE>'):] for x in       type_pattern.findall(raw_10k)]

Create a Dictionary for the 10-K

In the code below, we will create a dictionary which has the key 10-K and as value the contents of the 10-K section found above. To do this, we will create a loop, to go through all the sections found above, and if the section type is 10-K then save it to the dictionary. Use the indices in doc_start_is and doc_end_isto slice the raw_10k file.

In [23]:
doc_types

[]

In [22]:
document = {}

# Create a loop to go through each section type and save only the 10-K section in the dictionary
for doc_type, doc_start, doc_end in zip(doc_types, doc_start_is, doc_end_is):
    if doc_type == '10-K':
        document[doc_type] = raw_10k[doc_start:doc_end]

document

{}

In [19]:
# display an excerpt from the document
document['10-K'][0:500]

KeyError: '10-K'


## STEP 3 : Apply REGEXes to find Item 1A, 7, and 7A under 10-K Section
The items in this document can be found in four different patterns. For example Item 1A can be found in either of the following patterns:

>Item 1A

>Item&#160;1A

>Item&nbsp;1A

ITEM 1A

In the code below we will write a single regular expression that can match all four patterns for Items 1A, 7, and 7A. Then use the .finditer() method to match the regex to document['10-K'].

Note that Item 1B & Item 8 are added to find out end of section Item 1A & Item 7A subsequently.

In [20]:
# Write the regex
regex = re.compile(r'(>Item(\s|&#160;|&nbsp;)(1A|1B|7A|7|8)\.{0,1})|(ITEM\s(1A|1B|7A|7|8))')

# Use finditer to math the regex
matches = regex.finditer(document['10-K'])

# Write a for loop to print the matches
for match in matches:
    print(match)

KeyError: '10-K'